In [3]:
# pip install -U ddgs

In [4]:
from langchain_google_genai import GoogleGenerativeAI
from dotenv import load_dotenv
import os
import time

load_dotenv()
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

model = GoogleGenerativeAI(model="gemini-3.1-flash-lite")
model.invoke("최근 로제가 발표한 신곡은 무엇인가요?")

"로제가 최근 발표한 신곡은 브루노 마스(Bruno Mars)와 함께 부른 **'APT.' (아파트)**입니다.\n\n이 곡은 2024년 10월 18일에 공개되었으며, 한국의 술자리 게임인 '아파트 게임'에서 착안한 밝고 경쾌한 분위기의 노래로 전 세계적으로 큰 인기를 끌고 있습니다. 이 곡은 로제가 오는 12월 6일 발매할 첫 정규 앨범 'rosie'의 선공개 싱글입니다."

In [5]:
from langchain_community.tools import DuckDuckGoSearchResults

search = DuckDuckGoSearchResults(results_seperator=";\n")
docs = search.invoke("최근 로제가 발표한 신곡은 무엇인가요?")

print(docs)

C:\Users\bny64\AppData\Local\Temp\ipykernel_7756\592159615.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools import DuckDuckGoSearchResults


snippet: 지난해 10월 로제가 세계적 가수 브루노 마스와 함께 발표한 곡 'APT.'(아파트)가 최근 빌보드 메인 차트인 '핫 100'에서 2주 연속 3위를 차지하며 대중적 인기를 과시하고 있기 때문이다., title: 2년간 K팝 잊은 그래미…내년 로제가 자존심 회복시킬까 [N초점] - 뉴스1, link: https://www.news1.kr/entertain/music/5682705, snippet: [사진 = ROSÉ & Bruno Mars - APT. (Official MV)]. [이코노미 트리뷴 = 김용현 기자] 최근 국내에서 이른바 ‘로제’ 바람이 거세게 불고 있다. 4인조 걸그룹 ‘블랙핑크’ 메인보컬 로제가 최근 발간한 싱글 앨범 수록곡 ‘아파트(APT) 얘기다., title: 블랙핑크 로제가 쏘아올린 ‘한국판 스위프트노믹스’ | 이코노미 트리뷴, link: https://economytribune.co.kr/View.aspx?No=3422593, snippet: 로제가 지난해 10월 발표한 ‘아파트’ 역시 여전히 차트를 지켜, 솔로곡 총 2곡을 나란히 차트에 올렸다., title: [스경X이슈] “‘아파트’ 133억 수익” 로제, ‘메시’도 英 차트 진입, link: https://sports.khan.co.kr/article/202505171019003, snippet: 걸그룹 블랙핑크 멤버 로제가 발표한 신곡 ‘아파트(APT.)’의 글로벌 인기에 덩달아 국내 주류 업체인 하이트진로의 주가가 24일 6% 급등했다., title: 로제 신곡 ‘아파트’ 덕분… 하이트진로 6% 급등 | 조선일보, link: https://www.chosun.com/economy/money/2024/10/25/EKPTTE7YFZGADLUTUG644DSPKM/


In [6]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

question_answering_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "아래 context에 기반하여 사용자의 질문에 답변하라.:\n\n{context}"),
        MessagesPlaceholder(variable_name="messages"),
    ]
)

document_chain = question_answering_prompt | model

In [7]:
from langchain_community.chat_message_histories import ChatMessageHistory

# 채팅 메시지를 저장할 메모리 객체 생성
chat_history = ChatMessageHistory()

# 사용자 질문을 메모리에 저장
chat_history.add_user_message("요즘 로제가 발표한 신곡은 무엇인가요?")

# 문서 검색하고 답변 생성
answer = document_chain.invoke({"messages": chat_history.messages, "context": docs})

# 생성된 답변을 메모리에 저장
chat_history.add_ai_message(answer)
print(answer)

로제가 지난해 10월 브루노 마스와 함께 발표한 신곡의 제목은 **'APT.'(아파트)**입니다.


In [8]:
# DuckDuckGo API 래퍼를 사용하여 검색할 때 검색 매개변수를 설정하는 클래스 import
from langchain_community.utilities import DuckDuckGoSearchAPIWrapper

# 한국 지역("kr-kr")을 기준, 최근 일주일("w") 내의 검색 결과를 가져오도록 초기화
wrapper = DuckDuckGoSearchAPIWrapper(region="kr-kr", time="w")

In [9]:
# 검색 기능을 위한 DuckDuckGoSearchResults 초기화
search = DuckDuckGoSearchResults(
    api_wrapper=wrapper,  # 앞에서 정의한 API 래퍼를 사용
    source="news",  # 뉴스 소스에서만 검색하도록 지정
    results_separator=";\n",  # 결과 항목 사이에 구분자 사용(세미콜론과 줄 바꿈)
)

In [10]:
# "로제의 신곡 APT에 대한 반응"을 검색하고 결과를 docs에 저장
docs = search.invoke("로제의 신곡 APT에 대한 반응")

# 검색 결과 출력
print(docs)

snippet: 3 days ago - "APT." is a song by New Zealand and South Korean singer Rosé and American singer-songwriter Bruno Mars. It was released through The Black Label and Atlantic Records on 18 October 2024, as the lead single from Rosé's debut studio album, Rosie (2024). "APT." marked Rosé's first solo single ..., title: APT. (song) - Wikipedia, link: https://en.wikipedia.org/wiki/APT._(song);
snippet: 1 week ago - 로제의 이 독창적인 그루브와 목소리는 테디가 작곡한 블랙핑크 곡들이나 팝송과 만나면 시너지가 좋은 편인데, 이것은 로제의 보컬이 스탠다드한 기본기 탄탄한 발성보다 YG엔터테인..., title: 로제(BLACKPINK) - 나무위키, link: https://namu.wiki/w/로제(BLACKPINK);
snippet: 1 week ago - [서울=뉴시스]박영환 기자 = 로제와 브루노 마스의 히트곡 ‘아파트(APT.)’를 비롯해 세계 각국에서 발표된 약 500곡에서 서로 비슷한 네 음짜리 선율이 발견됐다는 분석이 나왔다. 덴마크 음악가 칼 마틴(27)이 수개월간 곡을 ..., title: 로제 '아파트' 속 그 4개 음…세계 각국 72곡서 닮은 선율 찾았다 :: 공감언론 뉴시스 ::, link: https://www.newsis.com/view/NISX20260728_0003726426;
snippet: 2 days ago - BLACKPINK’s Rosé and Bruno Mars’ “APT.” continues its record-breaking run on YouTube! On the morning of July 3

In [13]:
# DuckDuckGo를 이용해 ytn.co.kr 웹 사이트에서 로제의 신곡 APT에 대한 분석을 검색
docs = search.invoke("site:ytn.co.kr 로제의 신곡 APT에 대한 분석")

docs

'snippet: 5 days ago · 일본을 넘어 국내에서도 탄탄한 팬덤을 구축한 4인조 밴드 오피셜히게단디즘(OFFICIAL HIGE DANDISM, 이하 히게단)이 다시 서울을 찾는다. 첫 내한 공연을 단 10여 분 만..., title: [가요]日 넘어 K팝 팬도 사로잡은 히게단, 8월 서울 뜬다 | YTN, link: https://star.ytn.co.kr/_sn/0117_202607290752344732;\nsnippet: 4 days ago · 카카오 노사, 임금협상 잠정 합의...곧 찬반투표 진행 LG전자 2분기 최대 실적...매출 23.8조·영업익 1.6조 [속보] 코스피, 1.23% 내린 5,593.56에 장마감, title: [경제]삼성전자 반도체 영업익 89조..."메모리 공급난 지속" | YTN, link: https://www.ytn.co.kr/_ln/0102_202607301642549433;\nsnippet: 4 days ago · 삼성전자가 인공지능 반도체 호황에 힘입어 올해 2분기 90조 원에 달하는 영업이익을 거뒀습니다.실적 대부분이 반도체 사업에 집중된 가운데 ..., title: [경제]삼성전자 반도체로만 89.2조 벌었다...완제품 첫 적자 | YTN, link: https://www.ytn.co.kr/_ln/0102_202607300958115639;\nsnippet: 5 days ago · * 아래 텍스트는 실제 방송 내용과 차이가 있을 수 있으니 보다 정확한 내용은 방송으로 확인하시기 바랍니다. 인용 시 [YTN 뉴스퀘어 2PM] 명시해 ..., title: [Y녹취록]"고급 기술 걱정 없다, 다만"...중국 반도체 긴장하는 이유..., link: https://www.ytn.co.kr/_ln/0134_202607291617326247'

In [14]:
# 검색 결과의 링크들을 저장할 빈 리스트 초기화
links = []

# 검색 결과를 세미콜론과 줄 바꿈 기준으로 분리하고, 각 결과 항목에서 링크 추출
for doc in docs.split(";\n"):
    print(doc)  # 각 검색 결과 항목을 출력하여 확인
    link = doc.split("link:")[1].strip()  # 각 항목에서 'link:' 이후의 URL 부분만 추출
    links.append(link)  # 추출한 링크를 리스트에 추가

# 모든 링크를 출력
print(links)

snippet: 5 days ago · 일본을 넘어 국내에서도 탄탄한 팬덤을 구축한 4인조 밴드 오피셜히게단디즘(OFFICIAL HIGE DANDISM, 이하 히게단)이 다시 서울을 찾는다. 첫 내한 공연을 단 10여 분 만..., title: [가요]日 넘어 K팝 팬도 사로잡은 히게단, 8월 서울 뜬다 | YTN, link: https://star.ytn.co.kr/_sn/0117_202607290752344732
snippet: 4 days ago · 카카오 노사, 임금협상 잠정 합의...곧 찬반투표 진행 LG전자 2분기 최대 실적...매출 23.8조·영업익 1.6조 [속보] 코스피, 1.23% 내린 5,593.56에 장마감, title: [경제]삼성전자 반도체 영업익 89조..."메모리 공급난 지속" | YTN, link: https://www.ytn.co.kr/_ln/0102_202607301642549433
snippet: 4 days ago · 삼성전자가 인공지능 반도체 호황에 힘입어 올해 2분기 90조 원에 달하는 영업이익을 거뒀습니다.실적 대부분이 반도체 사업에 집중된 가운데 ..., title: [경제]삼성전자 반도체로만 89.2조 벌었다...완제품 첫 적자 | YTN, link: https://www.ytn.co.kr/_ln/0102_202607300958115639
snippet: 5 days ago · * 아래 텍스트는 실제 방송 내용과 차이가 있을 수 있으니 보다 정확한 내용은 방송으로 확인하시기 바랍니다. 인용 시 [YTN 뉴스퀘어 2PM] 명시해 ..., title: [Y녹취록]"고급 기술 걱정 없다, 다만"...중국 반도체 긴장하는 이유..., link: https://www.ytn.co.kr/_ln/0134_202607291617326247
['https://star.ytn.co.kr/_sn/0117_202607290752344732', 'https://www.ytn.co.kr/_ln/0102_202607301642549433',

In [15]:
# 랭체인의 WebBaseLoader를 사용하여 웹 페이지의 내용 불러오기
from langchain_community.document_loaders import WebBaseLoader

# WebBaseLoader 객체를 생성. 'links'는 웹 페이지의 URL 목록을 담고 있는 변수
# bs_get_text_kwargs는 BeautifulSoup의 get_text() 메서드에 전달될 추가 인자
loader = WebBaseLoader(
    web_paths=links,  # 웹 페이지의 링크 목록을 지정
    bs_get_text_kwargs={
        "strip": True
    },  # 웹 페이지에서 텍스트를 가져올 때 앞뒤의 공백 제거
)

# 비동기로 웹 페이지의 내용을 로드하고, 각 문서를 page_contents 리스트에 추가
page_contents = []  # 각 웹 페이지의 내용을 저장할 리스트
async for doc in loader.alazy_load():
    page_contents.append(doc) #불러온 문서를 page_contents 리스트에 추가

# page_contents에 있는 각 웹 페이지의 내용 출력
for content in page_contents:
    print(content) # 웹 페이지의 내용 출력
    print('--------------------')

USER_AGENT environment variable not set, consider setting it to identify your requests.
Fetching pages: 100%|##########| 4/4 [00:00<00:00, 32.22it/s]


page_content='[가요]日 넘어 K팝 팬도 사로잡은 히게단, 8월 서울 뜬다  | YTN메뉴 바로가기본문 바로가기푸터 바로가기닫기YTNYTN브랜드채널YTN 회사소개INSIDE YTNYTN 사이언스YTN 라디오YTN2YTN dmbYTN world남산서울타워장애인 서비스제보LIVE로그인회원가입로그아웃회원정보변경정치경제사회전국국제문화스포츠연예비즈날씨이슈시리즈TV프로그램日 넘어 K팝 팬도 사로잡은 히게단, 8월 서울 뜬다검색검색하기닫기많이 본 뉴스LIVE공유공유하기닫기페이스북엑스밴드카카오톡복사하기전체메뉴YTN닫기홈최신뉴스LIVE제보하기뉴스정치과학경제문화사회스포츠전국연예국제비즈날씨게임이슈속보단독재난시리즈몇층이세요한방이슈짤막상식와이즈픽자막뉴스뉴스모아제보영상와이파일운세앵커리포트나이트포커스지금이뉴스이게웬날리지Y녹취록이슈톺TV프로그램프로그램별날짜별앵커 소개시청자의견장애인 서비스검색하기日 넘어 K팝 팬도 사로잡은 히게단, 8월 서울 뜬다2026.07.29. 오전 07:52.댓글글자크기설정글자 크기 설정닫기가가가가가공유하기공유하기닫기페이스북엑스밴드카카오톡복사하기인쇄하기이미지 확대 보기사진 제공 = AEG PresentsAD일본을 넘어 국내에서도 탄탄한 팬덤을 구축한 4인조 밴드 오피셜히게단디즘(OFFICIAL HIGE DANDISM, 이하 히게단)이 다시 서울을 찾는다. 첫 내한 공연을 단 10여 분 만에 매진시키며 국내 인기를 증명했던 히게단은 이번에는 KSPO DOME을 무대로 팬들과 만난다.히게단은 오는 8월 8일과 9일 서울 올림픽공원 KSPO DOME에서 ‘OFFICIAL HIGE DANDISM ASIA TOUR 2026 in Seoul’을 개최한다. 2024년 첫 내한 이후 약 20개월 만의 한국 공연이다.첫 내한 당시 티켓 오픈과 동시에 전석 매진을 기록한 히게단은 팬들의 요청에 힘입어 추가 공연까지 진행했다. ‘도쿄 리벤저스’, ‘스파이 패밀리’, ‘100미터’ 등 인기 작품의 테마곡을 통해 대중적인 인지도를 쌓은 데 이어 누구나 공감할 수 있는 가사와 감

In [16]:
import requests
from bs4 import BeautifulSoup


# 주어진 URL에서 기사 텍스트를 가져오는 함수
def get_article_text(url):
    try:
        # URL에 GET 요청을 보냄
        response = requests.get(url)
        # 요청이 성공하지 못하면 예외를 발생시킴
        response.raise_for_status()

        # BeautifulSoup을 사용하여 HTML 내용을 파싱
        soup = BeautifulSoup(response.content, "html.parser")

        # 클래스가 'story-news article'인 <article> 태그 찾기
        article = soup.find("article", class_="story-news article")

        # 기사를 찾았다면 그 텍스트를 반환
        if article:
            return article.get_text(strip=True)
        else:
            try:
                if soup.find("article"):
                    return soup.find("article").get_text(strip=True)
                elif soup.find("div", id="CmAdContent"):
                    return soup.find("div", id="CmAdContent").get_text(strip=True)
            except:
                return "기사 내용을 찾을 수 없습니다."

    # 요청이 실패할 경우 예외 처리
    except requests.exceptions.RequestException as e:
        return f"URL을 가져오는 중 오류 발생: {e}"

In [17]:
# URL 목록의 각 링크를 반복하면서 기사 텍스트 출력
articles = []  # 가져온 내용을 리스트에 담는 변수 선언
for link in links:
    print(f"URL: {link}\n")
    article_text = get_article_text(link)
    print(f"Content:\n{article_text}")
    print("-----------------------------------")
    articles.append(article_text)

URL: https://star.ytn.co.kr/_sn/0117_202607290752344732

Content:
일본을 넘어 국내에서도 탄탄한 팬덤을 구축한 4인조 밴드 오피셜히게단디즘(OFFICIAL HIGE DANDISM, 이하 히게단)이 다시 서울을 찾는다. 첫 내한 공연을 단 10여 분 만에 매진시키며 국내 인기를 증명했던 히게단은 이번에는 KSPO DOME을 무대로 팬들과 만난다.히게단은 오는 8월 8일과 9일 서울 올림픽공원 KSPO DOME에서 ‘OFFICIAL HIGE DANDISM ASIA TOUR 2026 in Seoul’을 개최한다. 2024년 첫 내한 이후 약 20개월 만의 한국 공연이다.첫 내한 당시 티켓 오픈과 동시에 전석 매진을 기록한 히게단은 팬들의 요청에 힘입어 추가 공연까지 진행했다. ‘도쿄 리벤저스’, ‘스파이 패밀리’, ‘100미터’ 등 인기 작품의 테마곡을 통해 대중적인 인지도를 쌓은 데 이어 누구나 공감할 수 있는 가사와 감성적인 멜로디로 일본을 넘어 글로벌 음악 팬들의 사랑을 받고 있다.히게단은 일본 아티스트 최초로 애플 뮤직 어워드 ‘올해의 일본 아티스트상’을 수상했으며, 2025년에는 MTV VMAJ에서 ‘Rejoice’로 최우수 앨범상을 받았다. 스포티파이 팔로워 750만 명, 공식 유튜브 구독자 360만 명을 넘어선 데 이어 지난해 12월에는 일본 내 누적 스트리밍 100억 회를 돌파하며 음원 강자로서의 존재감도 과시했다.특히 히게단의 인기를 더욱 견고하게 만든 것은 뛰어난 라이브 역량이다. 2025년 스타디움 공연 ‘OFFICIAL HIGE DANDISM LIVE at STADIUM 2025’와 라이브하우스 투어 ‘OFFICIAL HIGE DANDISM One-man tour FOUR-RE’을 연이어 진행하며 규모를 가리지 않는 공연 역량을 보여줬다. 스타디움 공연 실황 영화도 일본과 대만에 이어 오는 4월 국내 개봉을 앞두고 있다. 8월 서울 공연에 앞서 히게단의 압도적인 라이브를 스크린으로 먼저 만날 